# RAG in Generative AI — Complete Jupyter Notebook

**RAG = Retrieval-Augmented Generation**

This notebook explains RAG in simple English using a real-world HR knowledge-base example and then builds the major components with Python:

- Document loading
- Chunking
- Chunk overlap
- Embeddings
- Cosine similarity
- Vector search
- Top-K retrieval
- Re-ranking
- Context construction
- Prompt construction
- LLM generation
- Complete RAG pipeline
- Hybrid search
- Metadata filtering
- RAG evaluation
- RAG vs Fine-Tuning
- Production architecture


## 1. What is RAG?

RAG stands for **Retrieval-Augmented Generation**.

The simple idea is:

> Retrieve relevant information first, give that information to the LLM, and then let the LLM generate an answer.

Traditional LLM:

```text
User Question → LLM → Answer
```

RAG:

```text
User Question
      ↓
   Retrieval
      ↓
Relevant Context
      ↓
Question + Context
      ↓
     LLM
      ↓
   Answer
```

### Real-world example

Imagine a company has an internal HR handbook containing annual leave, remote work, insurance, and travel policies.

An employee asks:

> How many annual leave days do employees get?

The LLM does not automatically know the company's private policy.

RAG searches the company's documents, retrieves:

> Employees receive 25 days of annual leave every year.

That text is placed into the LLM prompt, and the LLM generates the final answer.


## 2. Why is RAG Important?

LLMs may:

- Not know private company information
- Have outdated knowledge
- Hallucinate
- Lack access to internal documents and databases

RAG connects an LLM to external knowledge without necessarily retraining the LLM.

Common applications:

- HR assistants
- Customer support
- Legal document assistants
- Financial research
- Healthcare knowledge systems
- Developer documentation assistants
- Enterprise search


## 3. Benefits and Challenges

### Benefits

1. Uses private/domain-specific data
2. Data can be updated without retraining the LLM
3. Can ground answers in source documents
4. Usually simpler than fine-tuning for knowledge access
5. Works with PDFs, databases, websites, APIs, SharePoint, and cloud storage

### Challenges

1. Poor chunking can hurt retrieval
2. Poor embeddings can hurt semantic search
3. Initial retrieval may return noisy results
4. Re-ranking adds latency
5. LLMs can still hallucinate
6. Large document collections require scalable search
7. Access control and data security must be handled carefully


## 4. RAG Pipeline

### Offline / indexing stage

```text
Documents
   ↓
Text Extraction
   ↓
Cleaning
   ↓
Chunking
   ↓
Embeddings
   ↓
Vector Database
```

### Online / query stage

```text
User Question
      ↓
Query Embedding
      ↓
Vector Search
      ↓
Cosine Similarity / ANN Search
      ↓
Top-K Candidates
      ↓
Re-Ranking
      ↓
Top-N Relevant Chunks
      ↓
Prompt + Context
      ↓
LLM
      ↓
Final Answer
```


# Part 1 — Example Knowledge Base

For learning, we will use a fictional company HR knowledge base.

In production, these documents could come from PDFs, DOCX, databases, websites, SharePoint, S3, or Azure Blob Storage.


In [ ]:
documents = [
    {
        "id": "hr_001",
        "title": "Annual Leave Policy",
        "text": (
            "Employees receive 25 days of annual leave every year. "
            "Leave requests must be submitted through the HR portal. "
            "Managers should approve or reject leave requests within 3 working days."
        )
    },
    {
        "id": "hr_002",
        "title": "Remote Work Policy",
        "text": (
            "Employees can work remotely up to 3 days per week. "
            "Remote work must be approved by the employee's manager. "
            "Employees working remotely must follow company security policies."
        )
    },
    {
        "id": "hr_003",
        "title": "Health Insurance Policy",
        "text": (
            "Employees receive health insurance benefits after joining the company. "
            "The insurance plan covers the employee and eligible dependents. "
            "Additional benefits are available to employees with more than 5 years of service."
        )
    },
    {
        "id": "hr_004",
        "title": "Travel Expense Policy",
        "text": (
            "Employees can claim travel expenses for approved business trips. "
            "Expense claims must be submitted within 30 days of the business trip. "
            "Receipts are required for eligible expenses."
        )
    },
    {
        "id": "hr_005",
        "title": "Parental Leave Policy",
        "text": (
            "Eligible employees can request parental leave according to company policy. "
            "Parental leave requests must be submitted through the HR portal before the leave begins."
        )
    }
]

print("Documents:", len(documents))


# Part 2 — Document Loading

In a real RAG application, loading may look like:

```text
PDF → PDF loader
DOCX → Word loader
Website → Web loader
Database → SQL query
SharePoint → API
Blob/S3 → Cloud storage loader
```

Our example documents are already loaded into Python.


# Part 3 — Text Chunking

Large documents are split into smaller pieces called **chunks**.

Why?

- Better retrieval precision
- Smaller prompts
- Lower token usage
- Easier vector search

There is no universal perfect chunk size.


In [ ]:
def chunk_text(text, chunk_size=150, overlap=30):
    if chunk_size <= overlap:
        raise ValueError("chunk_size must be greater than overlap")

    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])

        if end >= len(text):
            break

        start = end - overlap

    return chunks


all_chunks = []

for doc in documents:
    chunks = chunk_text(doc["text"], chunk_size=150, overlap=30)

    for i, chunk in enumerate(chunks):
        all_chunks.append({
            "chunk_id": f"{doc['id']}_chunk_{i}",
            "document_id": doc["id"],
            "title": doc["title"],
            "text": chunk
        })

print("Total chunks:", len(all_chunks))


In [ ]:
for chunk in all_chunks:
    print("\n", chunk["chunk_id"])
    print(chunk["text"])


## Why Chunk Overlap?

Suppose a sentence crosses a chunk boundary.

Without overlap:

```text
Chunk 1: employee must submit leave
Chunk 2: requests through the HR portal
```

With overlap, some text is repeated:

```text
Chunk 1: employee must submit leave requests
Chunk 2: submit leave requests through the HR portal
```

Overlap helps preserve context.

The trade-off is more chunks and potentially more storage/token usage.


# Part 4 — Embeddings

An embedding converts text into a numerical vector.

```text
"How many annual leave days do I get?"
                 ↓
          Embedding Model
                 ↓
[0.12, -0.44, 0.81, 0.22, ...]
```

Semantically similar text should generally produce nearby vectors.

For example:

```text
"How many vacation days do I have?"
"What is the annual leave allowance?"
```

A real embedding model captures semantic similarity much better than word counting.

The next example uses a **toy embedding** only to teach the mechanics.


In [ ]:
from collections import Counter
import math

def build_vocabulary(texts):
    vocabulary = set()

    for text in texts:
        vocabulary.update(text.lower().split())

    return sorted(vocabulary)


def simple_embedding(text, vocabulary):
    words = text.lower().split()
    counts = Counter(words)
    return [counts[word] for word in vocabulary]


texts = [chunk["text"] for chunk in all_chunks]
vocabulary = build_vocabulary(texts)

print("Vocabulary size:", len(vocabulary))


In [ ]:
example_vector = simple_embedding(
    "employees receive annual leave",
    vocabulary
)

print("Vector length:", len(example_vector))
print(example_vector[:20])


## Production Embeddings

Production RAG systems normally use a pretrained embedding model such as:

- OpenAI embeddings
- Azure OpenAI embeddings
- Sentence Transformers
- Cohere embeddings
- Other domain-specific embedding models

Document embeddings and query embeddings must use compatible embedding spaces.


# Part 5 — Cosine Similarity

Once text is converted into vectors, we need to measure vector similarity.

**Cosine similarity** compares the direction of two vectors.

Conceptually:

```text
High cosine similarity
        ↓
Similar direction
        ↓
Likely similar meaning
```

Formula:

```text
cosine_similarity(A, B)
=
(A · B) / (||A|| × ||B||)
```

Typical mathematical range:

```text
-1 to +1
```

For many modern text embedding models, real-world scores often occupy a much narrower range.


In [ ]:
def cosine_similarity(a, b):
    dot_product = sum(x * y for x, y in zip(a, b))

    magnitude_a = math.sqrt(sum(x * x for x in a))
    magnitude_b = math.sqrt(sum(x * x for x in b))

    if magnitude_a == 0 or magnitude_b == 0:
        return 0.0

    return dot_product / (magnitude_a * magnitude_b)


In [ ]:
a = [1, 2, 3]
b = [1, 2, 3]
c = [3, 1, 0]

print("a vs b:", cosine_similarity(a, b))
print("a vs c:", cosine_similarity(a, c))


# Part 6 — Create Document Embeddings

Now create a vector for every chunk.

In production, these vectors would normally be stored in a vector database/search engine.


In [ ]:
chunk_vectors = [
    simple_embedding(chunk["text"], vocabulary)
    for chunk in all_chunks
]

print("Created vectors:", len(chunk_vectors))


# Part 7 — Query Embedding

The user query must also be converted into the same vector space.

```text
User Question
     ↓
Same Embedding Model
     ↓
Query Vector
```


In [ ]:
query = "How many annual leave days do employees get?"

query_vector = simple_embedding(
    query,
    vocabulary
)

print("Query vector length:", len(query_vector))


# Part 8 — Vector Search

We compare the query vector against every chunk vector.

```text
Query Vector
     ↓
Compare with all chunk vectors
     ↓
Cosine similarity scores
     ↓
Sort by score
     ↓
Top-K candidates
```

In a real vector database, approximate nearest-neighbor algorithms are usually used for scalability instead of manually comparing every vector.


In [ ]:
def vector_search(query, top_k=5):
    query_vector = simple_embedding(query, vocabulary)

    results = []

    for chunk, vector in zip(all_chunks, chunk_vectors):
        score = cosine_similarity(query_vector, vector)

        results.append({
            "chunk_id": chunk["chunk_id"],
            "document_id": chunk["document_id"],
            "title": chunk["title"],
            "text": chunk["text"],
            "cosine_score": score
        })

    results.sort(
        key=lambda x: x["cosine_score"],
        reverse=True
    )

    return results[:top_k]


candidates = vector_search(query, top_k=5)

for result in candidates:
    print(
        f"{result['cosine_score']:.4f} | "
        f"{result['title']} | "
        f"{result['text']}"
    )


# Part 9 — Top-K Retrieval

Imagine a vector database has one million chunks.

We do not send all one million to the LLM.

Instead:

```text
1,000,000 chunks
      ↓
Vector Search
      ↓
Top 20 candidates
      ↓
Re-ranker
      ↓
Top 3 chunks
      ↓
LLM
```

The initial retrieval stage is usually optimized for speed and recall.


# Part 10 — Re-Ranking

**Re-ranking is a second-stage retrieval step.**

Initial vector retrieval is fast, but the ranking may not be perfect.

So we use:

```text
Query
  ↓
Embedding
  ↓
Vector Search
  ↓
Top-K candidates
  ↓
Re-Ranker
  ↓
Final Top-N chunks
```

A common re-ranking approach uses a **cross-encoder**.

The cross-encoder receives the query and candidate document together:

```text
[Query, Candidate Chunk]
          ↓
      Cross-Encoder
          ↓
     Relevance Score
```

This is more precise but usually slower than vector retrieval, so we apply it to only a small candidate set.


## Cosine Similarity vs Re-Ranking

| Stage | Purpose | Typical approach |
|---|---|---|
| Embedding | Text → vector | Embedding model |
| Initial retrieval | Find candidates quickly | Vector database |
| Similarity | Vector closeness | Cosine similarity / ANN metric |
| Re-ranking | Better relevance ordering | Cross-encoder |
| Final context | Select chunks for LLM | Top-N |


# Part 11 — Real Cross-Encoder Re-Ranker

If needed in Jupyter/Colab:

```bash
pip install sentence-transformers
```

We use an example MS MARCO cross-encoder for demonstration.

For production, evaluate the reranker on your own domain, language, accuracy, and latency requirements.


In [ ]:
# Uncomment if needed:
# !pip install -q sentence-transformers

try:
    from sentence_transformers import CrossEncoder
    RERANKER_AVAILABLE = True
except ImportError:
    RERANKER_AVAILABLE = False

print("Reranker available:", RERANKER_AVAILABLE)


In [ ]:
if RERANKER_AVAILABLE:
    reranker = CrossEncoder(
        "cross-encoder/ms-marco-MiniLM-L-6-v2"
    )
else:
    reranker = None


In [ ]:
def rerank(query, candidates, top_k=3):
    if reranker is None:
        raise RuntimeError(
            "Install sentence-transformers first."
        )

    pairs = [
        [query, candidate["text"]]
        for candidate in candidates
    ]

    scores = reranker.predict(pairs)

    ranked = []

    for candidate, score in zip(candidates, scores):
        item = candidate.copy()
        item["rerank_score"] = float(score)
        ranked.append(item)

    ranked.sort(
        key=lambda x: x["rerank_score"],
        reverse=True
    )

    return ranked[:top_k]


In [ ]:
if RERANKER_AVAILABLE:
    final_chunks = rerank(
        query,
        candidates,
        top_k=3
    )

    for result in final_chunks:
        print(
            f"rerank={result['rerank_score']:.4f} | "
            f"cosine={result['cosine_score']:.4f} | "
            f"{result['title']} | "
            f"{result['text']}"
        )
else:
    final_chunks = candidates[:3]
    print("Reranker not installed; using top vector-search candidates.")


# Part 12 — Why Re-Ranking?

Suppose vector retrieval gives:

```text
1. Annual Leave Policy → 0.91
2. Health Insurance     → 0.87
3. Sick Leave           → 0.85
4. Remote Work          → 0.81
5. Travel Expenses      → 0.79
```

A reranker evaluates the query and each candidate more directly.

It may produce:

```text
Annual Leave Policy → very relevant
Sick Leave          → somewhat relevant
Health Insurance    → weakly relevant
Remote Work         → irrelevant
Travel Expenses     → irrelevant
```

The final top 2–3 chunks are then passed to the LLM.

### Key idea

**Cosine similarity:** "Which candidates are close in embedding space?"

**Re-ranker:** "Which candidate is actually most relevant to this exact query?"


# Part 13 — Build Context

Combine the final retrieved chunks into the context that will be supplied to the LLM.


In [ ]:
def build_context(results):
    return "\n\n".join(
        f"[Source: {r['title']}]\n{r['text']}"
        for r in results
    )


context = build_context(final_chunks)
print(context)


# Part 14 — Prompt Construction

The prompt should clearly tell the LLM to use the retrieved context.

A common grounding instruction is:

```text
Use only the provided context.
If the answer is not present, say that there is not enough information.
```

This reduces unsupported answers but does not mathematically guarantee zero hallucination.


In [ ]:
def build_prompt(question, context):
    return f"""
You are a helpful company HR assistant.

Answer the user's question using only the provided context.

Rules:
1. Do not invent information.
2. If the answer is not available in the context, say:
   "I don't have enough information in the provided documents."
3. Keep the answer concise.
4. Mention the relevant source when appropriate.

Context:
{context}

User Question:
{question}

Answer:
""".strip()


prompt = build_prompt(query, context)
print(prompt)


# Part 15 — LLM Generation

Now send the prompt to an LLM.

The exact API depends on your provider.

Conceptually:

```python
response = llm.generate(prompt)
```

Example using an OpenAI-compatible client:

```python
from openai import OpenAI

client = OpenAI()

response = client.responses.create(
    model="YOUR_MODEL",
    input=prompt
)

print(response.output_text)
```

The architecture is:

```text
Retrieved Context
       +
User Question
       ↓
    Prompt
       ↓
      LLM
       ↓
Generated Answer
```


# Part 16 — Complete Retrieval Function

Now combine vector retrieval and re-ranking.


In [ ]:
def retrieve_for_rag(question, candidate_k=5, final_k=3):
    # Stage 1: fast candidate retrieval
    candidates = vector_search(
        question,
        top_k=candidate_k
    )

    # Stage 2: precise re-ranking
    if RERANKER_AVAILABLE:
        final_results = rerank(
            question,
            candidates,
            top_k=final_k
        )
    else:
        final_results = candidates[:final_k]

    return final_results


def create_rag_prompt(question):
    results = retrieve_for_rag(
        question,
        candidate_k=5,
        final_k=3
    )

    context = build_context(results)
    prompt = build_prompt(question, context)

    return {
        "question": question,
        "results": results,
        "context": context,
        "prompt": prompt
    }


rag_result = create_rag_prompt(
    "How many annual leave days do employees get?"
)

print(rag_result["context"])


# Part 17 — Complete RAG Architecture

```text
                         DOCUMENTS
                            ↓
                        Chunking
                            ↓
                       Embeddings
                            ↓
                     Vector Database
                            │
                            │
                            ▼
USER QUERY → Query Embedding
                            ↓
                      Vector Search
                            ↓
                   Cosine Similarity
                            ↓
                    Top-K Candidates
                            ↓
                       Re-Ranker
                            ↓
                    Top-N Relevant
                       Chunks
                            ↓
                   Context Building
                            ↓
                  Prompt Construction
                            ↓
                           LLM
                            ↓
                   Grounded Answer
                            ↓
                     Source Citations
```

This is the core RAG architecture to remember.


# Part 18 — Metadata Filtering

A vector database can store metadata with every chunk.

Example:

```python
{
    "department": "HR",
    "country": "India",
    "document_type": "policy",
    "year": 2026
}
```

If a user asks about India's HR policy, we can filter:

```text
All documents
     ↓
country = India
     ↓
Vector Search
     ↓
Re-ranking
```

Metadata filtering can improve relevance, security, and efficiency.


In [ ]:
example_chunk = {
    "text": "Employees receive 25 days of annual leave.",
    "metadata": {
        "department": "HR",
        "country": "India",
        "document_type": "policy",
        "year": 2026
    }
}

print(example_chunk)


# Part 19 — Hybrid Search

Vector search is excellent for semantic meaning.

Keyword search is often excellent for exact terms, identifiers, product codes, error codes, and names.

Example:

```text
INC-48291
```

A practical RAG system may combine:

```text
Keyword Search
      +
Vector Search
      ↓
Hybrid Retrieval
      ↓
Re-Ranking
```

This can be particularly useful for enterprise search.


# Part 20 — Query Transformation

The user's original query is not always optimal for retrieval.

Example:

```text
"What happens if I want to work from home?"
```

A query transformation step might produce:

```text
"remote work policy employee work from home eligibility"
```

Advanced approaches include:

- Query rewriting
- Query expansion
- Multi-query retrieval
- Query decomposition
- HyDE


# Part 21 — RAG vs Fine-Tuning

### RAG

Use RAG when you need the model to access external/private/domain-specific information.

```text
Need new knowledge?
       ↓
      RAG
```

### Fine-Tuning

Use fine-tuning when you want to change model behavior, style, or task performance.

```text
Need different behavior?
       ↓
   Fine-tuning
```

Sometimes production systems use both.


# Part 22 — Hallucination and Grounding

RAG can reduce hallucination, but it cannot guarantee zero hallucination.

Useful techniques:

- Clear grounding instructions
- Good retrieval
- Re-ranking
- Context filtering
- Source citations
- Answer validation
- Evaluation datasets
- Guardrails

A useful response can include:

```text
Answer:
Employees receive 25 days of annual leave.

Source:
Annual Leave Policy
```

Tracking document IDs and chunk IDs makes answers auditable.


# Part 23 — Production RAG Architecture

A realistic enterprise architecture may look like:

```text
PDF / DOCX / Web / DB / SharePoint
              ↓
       Document Processing
              ↓
       Text Extraction / OCR
              ↓
            Chunking
              ↓
          Embeddings
              ↓
       Vector/Search Store
              │
              │
              ▼
User → API → Authentication
              ↓
       Query Understanding
              ↓
       Hybrid / Vector Search
              ↓
          Top-K Retrieval
              ↓
            Reranker
              ↓
       Context Filtering
              ↓
     Prompt Construction
              ↓
             LLM
              ↓
       Answer + Citations
```

Other production components may include:

- API Gateway
- Azure AI Search / Pinecone / Qdrant / pgvector / OpenSearch
- Redis caching
- Access-control filtering
- Observability
- Prompt management
- Rate limiting
- PII protection
- Audit logging
- Evaluation pipelines


# Part 24 — Important RAG Parameters

### Chunk size

How much text is placed in each chunk?

Too small:

```text
Not enough context
```

Too large:

```text
More noise
More tokens
Potentially worse retrieval
```

### Chunk overlap

How much text is repeated between adjacent chunks?

### Top-K

How many candidates are retrieved initially?

Example:

```text
Top-K = 20
```

### Final-K

How many chunks are passed after re-ranking?

Example:

```text
20 candidates
     ↓
  Reranker
     ↓
3 final chunks
```

### Similarity threshold

A system may reject candidates below a chosen relevance threshold.

The correct threshold must be evaluated on real data; there is no universal value.

### Context window

The LLM has a maximum input capacity. More context is not automatically better.


# Part 25 — RAG Evaluation

RAG has two major evaluation areas.

## Retrieval evaluation

Questions:

- Did we retrieve the correct document?
- Did the correct chunk appear in Top-K?
- Is the ranking good?

Common metrics:

- Recall@K
- Precision@K
- MRR
- NDCG

## Generation evaluation

Questions:

- Is the answer correct?
- Is it grounded in retrieved context?
- Is it relevant?
- Did it hallucinate?
- Are citations correct?

Common concepts:

- Faithfulness / groundedness
- Answer relevance
- Context relevance
- Citation correctness


# Part 26 — Interview Questions and Short Answers

### What is RAG?

Retrieval-Augmented Generation retrieves relevant external information and provides it to an LLM as context before generating an answer.

### What are embeddings?

Numerical vector representations of text used to represent semantic information.

### What is cosine similarity?

A mathematical measure of similarity between two vectors based on their direction.

### Why use a vector database?

To efficiently store and retrieve embeddings at scale.

### Why Top-K retrieval?

To reduce a huge document collection to a manageable set of candidates.

### Why re-ranking?

Because initial vector retrieval is optimized for speed, while a reranker can more precisely order a smaller set of candidates by query-document relevance.

### What is a cross-encoder?

A model that processes the query and candidate document together to produce a relevance score.

### What is the difference between retrieval and generation?

Retrieval finds relevant information. Generation uses that information to produce natural-language output.

### RAG vs Fine-Tuning?

RAG primarily supplies external knowledge. Fine-tuning primarily changes model behavior or task performance.


# Final Mental Model

Remember:

```text
RAG = Retrieve + Augment + Generate
```

And the practical pipeline:

```text
Documents
   ↓
Chunking
   ↓
Embeddings
   ↓
Vector Database
   ↓
User Query
   ↓
Query Embedding
   ↓
Vector Search
   ↓
Cosine Similarity / ANN
   ↓
Top-K Candidates
   ↓
Re-Ranking
   ↓
Top-N Chunks
   ↓
Prompt + Context
   ↓
LLM
   ↓
Grounded Answer + Citations
```

### The most important production insight

> **A powerful LLM cannot reliably answer from information that was never retrieved into its context.**

Therefore, RAG quality depends heavily on:

**good chunking → good embeddings → good retrieval → good re-ranking → good context → good generation.**
